# Advanced Python: AI Agents with LangChain & LangGraph

Build intelligent agents with LangChain for orchestration and LangGraph for complex agent workflows.

**Requires:** `pip install langchain langchain-openai langgraph python-dotenv`

**Setup:** Create `.env` file with `OPENAI_API_KEY=your_key_here`

## 1. LangChain Basics: Simple LLM Chain

# USAGE: Create a simple prompt-to-LLM pipeline
# This chain takes input, formats it with a template, sends to OpenAI, returns result

from langchain_openai import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Step 1: Initialize LLM (requires OPENAI_API_KEY)
# temperature: 0 = deterministic, 1 = creative
llm = OpenAI(temperature=0.7, max_tokens=100)

# Step 2: Create a prompt template with placeholders
# Input variable '{topic}' will be replaced with user input
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Write a short 2-line poem about {topic}"
)

# Step 3: Chain prompt + LLM together
chain = LLMChain(llm=llm, prompt=prompt)

# Step 4: Run the chain (requires valid API key)
# Uncomment to run: result = chain.run(topic="Python")
# Output: A poem about Python

print("LangChain chain created successfully")
print(f"Chain: Prompt -> LLM -> Output")

## 2. Conversation Memory: Stateful Chat

# USAGE: Keep conversation history across multiple turns
# Memory stores past exchanges so LLM understands context

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# Step 1: Create buffer memory to store conversation
# ConversationBufferMemory stores all messages (use ConversationSummaryMemory for large chats)
memory = ConversationBufferMemory()

# Step 2: Create conversation chain with memory
# llm: model to use
# memory: where to store conversation history
# verbose: True prints internal steps (for debugging)
conversation = ConversationChain(
    llm=OpenAI(temperature=0.7),
    memory=memory,
    verbose=False
)

# Step 3: Run multi-turn conversation (requires API key)
# Uncomment to run:
# response1 = conversation.run("My name is Alice")
# response2 = conversation.run("What's my name?")  # LLM remembers from response1

print("ConversationChain with memory configured")
print("Turn 1 input: 'My name is Alice'")
print("Turn 2 input: 'What's my name?' -> LLM recalls from memory")

## 3. Retrieval-Augmented Generation (RAG): Q&A from Documents

# USAGE: Answer questions based on custom documents
# Process: Load docs -> Split into chunks -> Embed -> Store -> Retrieve relevant chunks -> LLM answers

from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.chains import RetrievalQA

# Step 1: Sample document (in practice, load from file)
doc_text = """
Python is a high-level programming language created by Guido van Rossum.
It emphasizes code readability and simplicity. Python supports multiple
programming paradigms including object-oriented, procedural, and functional.
"""

# Step 2: Split document into smaller chunks for embedding
# chunk_size: max characters per chunk
# chunk_overlap: overlap between chunks (for context)
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=100,
    chunk_overlap=20
)
docs = splitter.split_text(doc_text)

# Step 3: Create embeddings (convert text to vectors)
# Requires OPENAI_API_KEY for embeddings
# embeddings = OpenAIEmbeddings()

# Step 4: Store embeddings in vector DB (FAISS is in-memory)
# vectordb = FAISS.from_texts(docs, embeddings)

# Step 5: Create QA chain with retrieval
# qa_chain = RetrievalQA.from_chain_type(
#     llm=OpenAI(temperature=0.7),
#     chain_type="stuff",  # concatenate retrieved docs
#     retriever=vectordb.as_retriever()
# )

# Step 6: Ask question
# answer = qa_chain.run("Who created Python?")

print("RAG pipeline configured")
print("Process: Docs -> Chunks -> Embeddings -> Vector DB -> Retrieval -> LLM")

## 4. LangGraph: Multi-Step Agent Workflow

# USAGE: Build complex agents with multiple decision points and tool use
# LangGraph orchestrates: Nodes (steps) -> Edges (transitions) -> State management

from langgraph.graph import StateGraph
from typing import TypedDict, List
import json

# Step 1: Define agent state (shared data)
class AgentState(TypedDict):
    """State passed between graph nodes"""
    input: str
    history: List[str]
    result: str

# Step 2: Define processing nodes (functions)
def process_input(state: AgentState) -> AgentState:
    """Node 1: Parse and validate input"""
    print(f"Processing: {state['input']}")
    state['history'].append(f"Parsed: {state['input']}")
    return state

def call_llm(state: AgentState) -> AgentState:
    """Node 2: Send to LLM"""
    print(f"Calling LLM for: {state['input']}")
    # llm_response = llm.predict(state['input'])
    state['history'].append("LLM called")
    state['result'] = "[LLM Response]"  # Placeholder
    return state

def format_output(state: AgentState) -> AgentState:
    """Node 3: Format final result"""
    print(f"Formatted result: {state['result']}")
    state['history'].append("Output formatted")
    return state

# Step 3: Create graph
graph = StateGraph(AgentState)

# Step 4: Add nodes (steps)
graph.add_node("process", process_input)
graph.add_node("llm", call_llm)
graph.add_node("format", format_output)

# Step 5: Add edges (define flow: process -> llm -> format)
graph.add_edge("process", "llm")
graph.add_edge("llm", "format")

# Step 6: Set entry and exit points
graph.set_entry_point("process")
graph.set_finish_point("format")

# Step 7: Compile graph
agent = graph.compile()

# Step 8: Run agent
# result = agent.invoke({"input": "Hello", "history": [], "result": ""})

print("LangGraph agent configured")
print("Flow: [process] -> [llm] -> [format]")

## 5. Tool Use: Give Agents Access to External Functions

# USAGE: LLM can call external tools (APIs, functions) to answer questions
# Example: Calculator, web search, database queries

from langchain.agents import tool

# Step 1: Define tools as functions with @tool decorator
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression. Use when asked to calculate."""
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

@tool
def search_knowledge_base(query: str) -> str:
    """Search internal knowledge base. Use for factual questions."""
    # In real app: query database or vector store
    return f"Found info about: {query}"

# Step 2: Create agent with tools
# from langchain.agents import initialize_agent, AgentType
# tools = [calculator, search_knowledge_base]
# agent = initialize_agent(
#     tools=tools,
#     llm=OpenAI(temperature=0),
#     agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#     verbose=True
# )

# Step 3: Run agent (LLM decides which tool to use)
# result = agent.run("What is 15 * 8?")
# -> LLM sees 'calculator' tool, calls it, returns 120

print("Tool definitions:")
print(f"- calculator: {calculator.description}")
print(f"- search_knowledge_base: {search_knowledge_base.description}")

---

**Setup Checklist:**
1. `pip install langchain langchain-openai langgraph python-dotenv`
2. Create `.env` with `OPENAI_API_KEY=sk-...`
3. Run cells to execute (requires API key and credits)

**Common Patterns:**
- **LLMChain:** Simple input -> template -> LLM -> output
- **ConversationChain:** Multi-turn with memory
- **RetrievalQA:** Q&A from documents (RAG)
- **LangGraph:** Complex workflows with decisions
- **Tool Use:** LLM choosing from available functions